# Feature Engineering I: Transformations, Encoding, Dates, Text and Geo

Notebook for **Chapter 5** of the *ds* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/ds/ch05.html

Tip: change a number, predict the output, then run the cell again.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

d = features(pd.Timestamp("2025-01-01"))       # helper from Chapter 3
tr, te = train_test_split(d, test_size=0.3, random_state=0, stratify=d.churn)
def fit_auc(Xtr, Xte):
    m = LogisticRegression(max_iter=3000).fit(Xtr, tr.churn)
    return round(roc_auc_score(te.churn, m.predict_proba(Xte)[:, 1]), 3)
base = ["orders_90d", "recency_days"]
print("AUC base (orders_90d, recency):", fit_auc(tr[base], te[base]))

AUC base (orders_90d, recency): 0.883


## 5.1 Numeric Transformations

In [ ]:
from sklearn.preprocessing import PowerTransformer
print("skew spend:", round(d.spend_90d.skew(), 2), "log1p:", round(np.log1p(d.spend_90d).skew(), 2),
      "sqrt:", round(np.sqrt(d.spend_90d).skew(), 2))
pt = PowerTransformer(method="yeo-johnson").fit(tr[["spend_90d"]])
print("yeo-johnson skew:", round(float(pd.Series(pt.transform(d[["spend_90d"]]).ravel()).skew()), 2),
      " lambda", round(pt.lambdas_[0], 3))
lo, hi = d.spend_90d.quantile([0.01, 0.99]); print("1%/99% quantile:", round(lo), round(hi), " max", round(d.spend_90d.max()))
clipped = d.spend_90d.clip(lo, hi)
print("clipped max", round(clipped.max()), " mean before/after", round(d.spend_90d.mean()), round(clipped.mean()))

skew spend: 2.17 log1p: -0.43 sqrt: 0.78
yeo-johnson skew: -0.03  lambda 0.161
1%/99% quantile: 207 24601  max 41357
clipped max 24601  mean before/after 5261 5193


In [ ]:
from sklearn.preprocessing import KBinsDiscretizer
kb = KBinsDiscretizer(n_bins=5, encode="onehot-dense", strategy="quantile").fit(tr[["recency_days"]])
print("quantile bin edges recency:", kb.bin_edges_[0].round(0))
Xb_tr = np.c_[tr[["orders_90d"]].values, kb.transform(tr[["recency_days"]])]
Xb_te = np.c_[te[["orders_90d"]].values, kb.transform(te[["recency_days"]])]
print("AUC with binned recency   :", fit_auc(Xb_tr, Xb_te))
for df in (tr, te):
    df["log_rec"] = np.log1p(df.recency_days)
    df["rec_ratio"] = df.recency_days / (90 / df.orders_90d)    # recency compared with the usual gap
print("AUC with log recency      :", fit_auc(tr[["orders_90d", "log_rec"]], te[["orders_90d", "log_rec"]]))
print("AUC + ratio feature (recency / usual gap):", fit_auc(tr[base + ["rec_ratio"]], te[base + ["rec_ratio"]]))

quantile bin edges recency: [ 1.  3.  7. 14. 34. 90.]
AUC with binned recency   : 0.871
AUC with log recency      : 0.88
AUC + ratio feature (recency / usual gap): 0.889


## 5.2 Encoding Categories (and Target Encoding Done Safely)

In [ ]:
print(d.city.value_counts().to_dict())
oh = pd.get_dummies(d[["plan", "channel"]], drop_first=True); print("one-hot columns:", list(oh.columns))
freq = d.city.map(d.city.value_counts(normalize=True)); print("frequency encoding Pune:", round(freq[d.city == "Pune"].iloc[0], 3))

{'Other': 526, 'Mumbai': 525, 'Bengaluru': 504, 'Delhi': 497, 'Pune': 264, 'Chennai': 246}
one-hot columns: ['plan_pro', 'channel_organic', 'channel_referral']
frequency encoding Pune: 0.103


In [ ]:
rng = np.random.default_rng(5)
d["pincode"] = rng.integers(0, 400, len(d)).astype(str)          # random: no signal
tr, te = train_test_split(d, test_size=0.3, random_state=0, stratify=d.churn)
means = tr.groupby("pincode").churn.mean(); gm = tr.churn.mean()
tr_naive = tr.pincode.map(means); te_naive = te.pincode.map(means).fillna(gm)

from sklearn.model_selection import KFold
def oof_te(train, test, col="pincode", k=5, smooth=20, seed=0):
    gm = train.churn.mean(); out = pd.Series(index=train.index, dtype=float)
    for a, b in KFold(k, shuffle=True, random_state=seed).split(train):
        s = train.iloc[a].groupby(col).churn.agg(["sum", "count"])
        enc = (s["sum"] + smooth * gm) / (s["count"] + smooth)
        out.iloc[b] = train.iloc[b][col].map(enc).fillna(gm).values
    s = train.groupby(col).churn.agg(["sum", "count"]); enc = (s["sum"] + smooth * gm) / (s["count"] + smooth)
    return out, test[col].map(enc).fillna(gm)

def single_auc(a_tr, a_te):
    m = LogisticRegression(max_iter=1000).fit(np.c_[a_tr], tr.churn)
    return (round(roc_auc_score(tr.churn, m.predict_proba(np.c_[a_tr])[:, 1]), 3),
            round(roc_auc_score(te.churn, m.predict_proba(np.c_[a_te])[:, 1]), 3))
print("levels:", d.pincode.nunique(), " mean rows per level (train):", round(len(tr) / 400, 1))
print("naive TE  train AUC, test AUC:", single_auc(tr_naive, te_naive))
a_tr, a_te = oof_te(tr, te)
print("OOF+smooth TE train AUC, test AUC:", single_auc(a_tr, a_te))
from sklearn.preprocessing import TargetEncoder
te_sk = TargetEncoder(target_type="binary", random_state=0)
print("sklearn TargetEncoder train, test AUC:",
      single_auc(te_sk.fit_transform(tr[["pincode"]], tr.churn).ravel(), te_sk.transform(te[["pincode"]]).ravel()))

levels: 396  mean rows per level (train): 4.5
naive TE  train AUC, test AUC: (0.812, 0.476)
OOF+smooth TE train AUC, test AUC: (0.531, 0.469)
sklearn TargetEncoder train, test AUC: (0.529, 0.474)


## 5.3 Dates, Cycles and Missing-Value Indicators

In [ ]:
cal = pd.DataFrame({"date": pd.date_range("2024-10-28", "2024-11-03")})
hol = pd.to_datetime(["2024-10-31", "2024-11-01"])        # example festival dates; verify with an official calendar
cal["dow"] = cal.date.dt.day_name().str[:3]
cal["is_weekend"] = (cal.date.dt.dayofweek >= 5).astype(int)
cal["is_holiday"] = cal.date.isin(hol).astype(int)
nxt = cal.date.map(lambda t: min([x for x in hol if x >= t], default=pd.NaT))
cal["days_to_next_holiday"] = (nxt - cal.date).dt.days
print(cal.to_string(index=False))

      date dow  is_weekend  is_holiday  days_to_next_holiday
2024-10-28 Mon           0           0                   3.0
2024-10-29 Tue           0           0                   2.0
2024-10-30 Wed           0           0                   1.0
2024-10-31 Thu           0           1                   0.0
2024-11-01 Fri           0           1                   0.0
2024-11-02 Sat           1           0                   NaN
2024-11-03 Sun           1           0                   NaN


In [ ]:
hrs = pd.date_range("2025-03-03", periods=24 * 14, freq="h")
h = hrs.hour.values
rng = np.random.default_rng(1)
load = 200 + 150 * np.sin(2 * np.pi * (h - 14) / 24) + rng.normal(0, 15, len(h))   # peak at 20:00
ev = pd.DataFrame({"hour": h, "load": load})
ev["sin_h"] = np.sin(2 * np.pi * h / 24); ev["cos_h"] = np.cos(2 * np.pi * h / 24)
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
m1 = LinearRegression().fit(ev[["hour"]], ev.load); print("R2 linear on raw hour :", round(r2_score(ev.load, m1.predict(ev[["hour"]])), 3))
m2 = LinearRegression().fit(ev[["sin_h", "cos_h"]], ev.load); print("R2 linear on sin/cos  :", round(r2_score(ev.load, m2.predict(ev[["sin_h", "cos_h"]])), 3))

R2 linear on raw hour : 0.388
R2 linear on sin/cos  : 0.984


In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
cv = StratifiedKFold(5, shuffle=True, random_state=0)
for cols in (["age_imp"], ["age_imp", "age_missing"], ["age_missing"]):
    s = cross_val_score(LogisticRegression(max_iter=2000), d[cols], d.churn, cv=cv, scoring="roc_auc")
    print(cols, s.round(3), s.mean().round(3))

['age_imp'] [0.563 0.548 0.532 0.505 0.479] 0.525
['age_imp', 'age_missing'] [0.574 0.606 0.579 0.552 0.522] 0.567
['age_missing'] [0.52  0.566 0.556 0.553 0.54 ] 0.547


In [ ]:
r = np.random.default_rng(11)
d["age_raw"] = d.age.copy()
d.loc[r.random(len(d)) < 0.07, "age_raw"] = -1
d.loc[r.random(len(d)) < np.where(d.churn == 1, 0.18, 0.05), "age_raw"] = np.nan
d["age_clean"] = d.age_raw.replace(-1, np.nan)          # -1 is a default code, so it becomes missing
d["age_missing"] = d.age_clean.isna().astype(int)
d["age_imp"] = d.age_clean.fillna(d.age_clean.median())

## 5.4 Text and Geo Features

In [ ]:
rs = np.random.default_rng(3)
refund_w = ["refund", "money back", "cancel order", "charged twice", "reimburse"]
other_w = ["login", "password", "delivery late", "app crash", "address change", "invoice copy", "update phone"]
filler = ["please", "help", "urgent", "hello", "thanks", "I am", "since morning", "again", "very", "support"]
rows = []
for i in range(3000):
    y = rs.random() < 0.3
    words = list(rs.choice(refund_w if y else other_w, 1)) + list(rs.choice(other_w, 1)) + list(rs.choice(filler, rs.integers(3, 9)))
    if y and rs.random() < 0.3: words = words[1:]
    rs.shuffle(words); rows.append((" ".join(words), int(y)))
tk = pd.DataFrame(rows, columns=["text", "refund"])

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
tk["n_chars"] = tk.text.str.len(); tk["n_words"] = tk.text.str.split().str.len()
tk["has_urgent"] = tk.text.str.contains("urgent").astype(int)
ttr, tte = train_test_split(tk, test_size=0.3, random_state=0)
cols = ["n_chars", "n_words", "has_urgent"]
m = LogisticRegression(max_iter=1000).fit(ttr[cols], ttr.refund)
print("text: length/count features AUC:", round(roc_auc_score(tte.refund, m.predict_proba(tte[cols])[:, 1]), 3))
tf = TfidfVectorizer(ngram_range=(1, 2), min_df=3).fit(ttr.text)
m = LogisticRegression(max_iter=1000).fit(tf.transform(ttr.text), ttr.refund)
print("text: TF-IDF AUC:", round(roc_auc_score(tte.refund, m.predict_proba(tf.transform(tte.text))[:, 1]), 3), " vocab size", len(tf.vocabulary_))
coef = pd.Series(m.coef_[0], index=tf.get_feature_names_out()).sort_values(ascending=False)
print("top words:", coef.head(5).round(2).to_dict())

text: length/count features AUC: 0.609
text: TF-IDF AUC: 0.99  vocab size 439
top words: {'reimburse': 4.96, 'refund': 4.76, 'money back': 2.68, 'back': 2.68, 'money': 2.68}


In [ ]:
def haversine(lat1, lon1, lat2, lon2, R=6371.0):
    p1, p2 = np.radians(lat1), np.radians(lat2); dphi = p2 - p1; dl = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))
print("Mumbai-Pune km:", round(float(haversine(19.0760, 72.8777, 18.5204, 73.8567)), 1))
print("Delhi-Bengaluru km:", round(float(haversine(28.6139, 77.2090, 12.9716, 77.5946)), 1))

Mumbai-Pune km: 120.2
Delhi-Bengaluru km: 1739.8


In [ ]:
g = np.random.default_rng(4)
centers = np.array([[19.07, 72.88], [28.61, 77.21], [12.97, 77.59]])
pts = np.vstack([c0 + g.normal(0, 0.15, (200, 2)) for c0 in centers])
wh = np.array([[19.2, 72.9], [28.5, 77.3]])
dist = np.min([haversine(pts[:, 0], pts[:, 1], w[0], w[1]) for w in wh], axis=0)
print("median km", round(float(np.median(dist)), 1), " Bengaluru-cluster median", round(float(np.median(dist[400:])), 0))
from sklearn.cluster import KMeans
km = KMeans(3, n_init=10, random_state=0).fit(pts)
print("KMeans cluster sizes:", np.bincount(km.labels_).tolist())
cell = (np.floor(pts[:, 0] * 10) / 10).astype(str) + "_" + (np.floor(pts[:, 1] * 10) / 10).astype(str)
print("grid cells of 0.1 degree:", len(set(cell)), "for", len(pts), "points")

median km 31.0  Bengaluru-cluster median 855.0
KMeans cluster sizes: [200, 200, 200]
grid cells of 0.1 degree: 151 for 600 points


## Chapter recap

## Practice set

Problem 5.P.1 (easy) Compute `log1p(19)`. **Answer:** ln(20) = 2.996.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/ds/ch05.html